In [18]:
import os, random, numpy as np, tensorflow as tf, pandas as pd

os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [4]:
base_dir = "dataset/chest_xray/chest_xray"   # change if needed
trainval_dir = os.path.join(base_dir, "train")  # we'll also include val manually below
val_dir_kaggle = os.path.join(base_dir, "val")
test_dir = os.path.join(base_dir, "test")

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
seed = 123

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ✅ replaces rescale=1./255
    validation_split=0.2,
    rotation_range=5,
    width_shift_range=0.02,
    height_shift_range=0.02,
    zoom_range=0.05,
    brightness_range=(0.9, 1.1),
    horizontal_flip=False
)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                 validation_split=0.2
                                )

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    trainval_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=True, seed=seed, subset="training"
)

val_generator = val_datagen.flow_from_directory(
    trainval_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False, subset="validation", seed=seed
)

test_generator = test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False
)


Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


## Build the Densenet121 transfer learning model
- Transfer learning gives better feautures immediately
- Faster convergence

The following function builds a model where DenseNet121 extracts features and a small head learns pneumonia vs normal, while initially freezing DenseNet so training is stable and doesn’t overfit.

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121

def build_densenet121(dropout=0.4):
    base_model = DenseNet121(
        weights="imagenet",
        include_top=False, #removes the final classification layer
        input_shape=IMG_SIZE + (3,)
    )
    # Freeze the base model which means the weights will not be updated during training
    base_model.trainable = False  # Stage 1: freeze backbone weights

    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = base_model(inputs, training=False)
    # Global average pooling to reduce each feature map to a single value
    x = layers.GlobalAveragePooling2D()(x)
    # Dropout for regularization
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy", #better than accuracy for imbalanced datasets
            tf.keras.metrics.AUC(name="auroc"),
            tf.keras.metrics.AUC(name="auprc", curve="PR")
        ]
    )
    return model

model = build_densenet121(dropout=0.4)
model.summary()


2025-12-29 19:00:07.237914: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-12-29 19:00:07.237950: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-12-29 19:00:07.237960: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-12-29 19:00:07.237978: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-12-29 19:00:07.237991: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,038,529 (26.85 MB)

 Trainable params: 1,025 (4.00 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

## Callbacks

In [7]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    ModelCheckpoint(
        "best_densenet121.keras",
        monitor="val_auprc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_auprc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_auprc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]


Monitoring AUPRC because the data is imbalaced and it refecs preision-recall trade-offs

Training is controlled using early stopping, learning rate reduction on plateau, and model checkpointing based on validation AUPRC to prevent overfitting and retain the best-performing model

### Stage 1 training (train head only)
- DenseNet1212 was trained on millions of images and its weight already encode edges, textures, shapes so we keep the pretrained netwrok frozen and train only the new layers that we added on top i.e the head.

In [8]:
history_stage1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks
)


Epoch 1/10


2025-12-29 19:01:27.321726: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step - accuracy: 0.7341 - auprc: 0.8383 - auroc: 0.6819 - loss: 0.5607
Epoch 1: val_auprc improved from None to 0.98972, saving model to best_densenet121.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 65s 419ms/step - accuracy: 0.8100 - auprc: 0.9342 - auroc: 0.8404 - loss: 0.4121 - val_accuracy: 0.9070 - val_auprc: 0.9897 - val_auroc: 0.9693 - val_loss: 0.2509 - learning_rate: 0.0010
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - accuracy: 0.8930 - auprc: 0.9817 - auroc: 0.9495 - loss: 0.2559
Epoch 2: val_auprc improved from 0.98972 to 0.99322, saving model to best_densenet121.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 45s 341ms/step - accuracy: 0.9068 - auprc: 0.9844 - auroc: 0.9575 - loss: 0.2341 - val_accuracy: 0.9214 - val_auprc: 0.9932 - val_auroc: 0.9798 - val_loss: 0.1994 - learning_rate: 0.0010
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step - accuracy: 0.9112 - auprc: 0.9861 - auroc: 0.9623 - loss: 0.2179
Epoch 3: val_auprc improved from 0.993

In [9]:
from utils import evaluate


In [10]:
results = []
results.append(evaluate(model, test_generator, name="DenseNet121", threshold=0.5, flip=False, class_weights=False))
results.append(evaluate(model, test_generator, name="DenseNet121", threshold=0.575, flip=False, class_weights=False))



=== DenseNet121 @ threshold 0.500 ===
[[167  67]
 [ 11 379]]
              precision    recall  f1-score   support

      NORMAL       0.94      0.71      0.81       234
   PNEUMONIA       0.85      0.97      0.91       390

    accuracy                           0.88       624
   macro avg       0.89      0.84      0.86       624
weighted avg       0.88      0.88      0.87       624

AUROC: 0.9615 | AUPRC: 0.9731
Macro F1: 0.8587 | Weighted F1: 0.8707
Sensitivity: 0.9718 | Specificity: 0.7137 | Precision: 0.8498

=== DenseNet121 @ threshold 0.575 ===
[[179  55]
 [ 15 375]]
              precision    recall  f1-score   support

      NORMAL       0.92      0.76      0.84       234
   PNEUMONIA       0.87      0.96      0.91       390

    accuracy                           0.89       624
   macro avg       0.90      0.86      0.88       624
weighted avg       0.89      0.89      0.89       624

AUROC: 0.9615 | AUPRC: 0.9731
Macro F1: 0.8755 | Weighted F1: 0.8853
Sensitivity: 0.9615 | 

Using DenseNet121, increasing the operating threshold from 0.5 to 0.575 reduced false positives by 18% while maintaining high sensitivity (96.2%). This resulted in improved precision, specificity, and F1 scores, with only a minor increase in false negatives. Given the higher clinical cost of missed pneumonia cases, the threshold of 0.575 was selected as the final operating point

In [11]:
model.load_weights("best_densenet121.keras")

base_model = model.layers[1]
base_model.trainable = True

# Freeze early layers, fine-tune last block
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auroc"),
        tf.keras.metrics.AUC(name="auprc", curve="PR")
    ]
)

history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks
)


Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step - accuracy: 0.9098 - auprc: 0.9887 - auroc: 0.9751 - loss: 0.2421
Epoch 1: val_auprc did not improve from 0.99707
131/131 ━━━━━━━━━━━━━━━━━━━━ 74s 446ms/step - accuracy: 0.9121 - auprc: 0.9887 - auroc: 0.9739 - loss: 0.2347 - val_accuracy: 0.9511 - val_auprc: 0.9968 - val_auroc: 0.9906 - val_loss: 0.1171 - learning_rate: 1.0000e-05
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step - accuracy: 0.9213 - auprc: 0.9888 - auroc: 0.9761 - loss: 0.2058
Epoch 2: val_auprc did not improve from 0.99707

Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
131/131 ━━━━━━━━━━━━━━━━━━━━ 46s 353ms/step - accuracy: 0.9312 - auprc: 0.9900 - auroc: 0.9775 - loss: 0.1794 - val_accuracy: 0.9521 - val_auprc: 0.9970 - val_auroc: 0.9912 - val_loss: 0.1124 - learning_rate: 1.0000e-05
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step - accuracy: 0.9398 - auprc: 0.9924 - auroc: 0.9812 - loss: 0.1506
Epoch 3: val_auprc improved

In [13]:
results = []
results.append(evaluate(model, test_generator, name="DenseNet121", threshold=0.5, flip=False, class_weights=False))
results.append(evaluate(model, test_generator, name="DenseNet121", threshold=0.575, flip=False, class_weights=False))




=== DenseNet121 @ threshold 0.500 ===
[[154  80]
 [  8 382]]
              precision    recall  f1-score   support

      NORMAL       0.95      0.66      0.78       234
   PNEUMONIA       0.83      0.98      0.90       390

    accuracy                           0.86       624
   macro avg       0.89      0.82      0.84       624
weighted avg       0.87      0.86      0.85       624

AUROC: 0.9654 | AUPRC: 0.9772
Macro F1: 0.8372 | Weighted F1: 0.8521
Sensitivity: 0.9795 | Specificity: 0.6581 | Precision: 0.8268

=== DenseNet121 @ threshold 0.575 ===
[[159  75]
 [ 10 380]]
              precision    recall  f1-score   support

      NORMAL       0.94      0.68      0.79       234
   PNEUMONIA       0.84      0.97      0.90       390

    accuracy                           0.86       624
   macro avg       0.89      0.83      0.84       624
weighted avg       0.87      0.86      0.86       624

AUROC: 0.9654 | AUPRC: 0.9772
Macro F1: 0.8442 | Weighted F1: 0.8580
Sensitivity: 0.9744 | 

- The model ranks pneumonia vs normal better after fine-tuning, AUPROC increased to 0.9654 and AUPRC to 0.9772 compared to stage 1
- Stage 2 with threshold 0.5 became more aggressive in catcing pneumonia compared to stage 1 as the sensitivity increases and specificity decreases
- AT our previous chosen 0.575 threshold, sensitivity is better (it has 5 less False negatives) but the specificuty is worsen (it has 20 fP) which means it favors catcihng pneumonia at the cost of more false alarms (declaring normal as pneumonia)

- Fine-tuning the DenseNet121 backbone further improved overall discrimination performance, as reflected by increased AUROC and AUPRC. At fixed operating thresholds, fine-tuning shifted the decision boundary toward higher sensitivity, reducing false negatives at the expense of increased false positives. This behavior is consistent with improved task-specific feature adaptation and suggests that threshold re-optimization is appropriate after fine-tuning

## Retune the threshold

In [14]:
model.load_weights("best_densenet121.keras")

In [15]:
val_generator.reset()
val_probs = model.predict(val_generator, verbose=0).ravel()
y_val = val_generator.classes

In [16]:
from utils import eval_thresholds


In [19]:
thresholds = np.linspace(0.3, 0.9, 61)

val_table = eval_thresholds(model, val_generator, thresholds)
val_table


,Threshold,Accuracy,Macro F1,Weighted F1,Specificity,Sensitivity,FN,FP
0,0.30,0.963,0.952,0.963,0.944,0.969,24,15
1,0.31,0.964,0.953,0.964,0.948,0.969,24,14
2,0.32,0.964,0.953,0.964,0.955,0.966,26,12
3,0.33,0.965,0.954,0.965,0.959,0.966,26,11
4,0.34,0.964,0.953,0.964,0.959,0.965,27,11
...,...,...,...,...,...,...,...,...
56,0.86,0.919,0.903,0.923,0.996,0.893,83,1
57,0.87,0.916,0.899,0.919,0.996,0.888,87,1
58,0.88,0.913,0.896,0.916,0.996,0.884,90,1
59,0.89,0.910,0.893,0.914,0.996,0.880,93,1


In [21]:
best = (
    val_table[val_table["Sensitivity"] >= 0.96]
    .sort_values("Specificity", ascending=False)
    .iloc[0]
)

best


Threshold       0.380
Accuracy        0.962
Macro F1        0.951
Weighted F1     0.962
Specificity     0.963
Sensitivity     0.961
FN             30.000
FP             10.000
Name: 8, dtype: float64

In [22]:
evaluate(
    model,
    test_generator,
    name="DenseNet121 Stage2",
    threshold=0.38
)



=== DenseNet121 Stage2 @ threshold 0.380 ===
[[139  95]
 [  6 384]]
              precision    recall  f1-score   support

      NORMAL       0.96      0.59      0.73       234
   PNEUMONIA       0.80      0.98      0.88       390

    accuracy                           0.84       624
   macro avg       0.88      0.79      0.81       624
weighted avg       0.86      0.84      0.83       624

AUROC: 0.9654 | AUPRC: 0.9772
Macro F1: 0.8086 | Weighted F1: 0.8274
Sensitivity: 0.9846 | Specificity: 0.5940 | Precision: 0.8017


{'Model': 'DenseNet121 Stage2',
 'Flip': None,
 'Class Weights': 'None',
 'Threshold': 0.38,
 'AUROC': 0.96537365768135,
 'AUPRC': 0.9772447496101226,
 'Accuracy': 0.8381410256410257,
 'Macro F1': 0.8086418441116012,
 'Weighted F1': 0.8274249964323775,
 'Sensitivity': 0.9846153846153847,
 'Specificity': 0.594017094017094,
 'Precision': 0.8016701461377871,
 'TN': 139,
 'FP': 95,
 'FN': 6,
 'TP': 384}

The baseline model at 0.575 threshold and transfer leaning model at threshold 0.38 has similar FP - 95 but FN has decreased from 14 to 6, which means baseling misses 14 penumonia cases whereas densent misses only 6, which could be considered as a great improvement specially in the field of medical.
Densent has higher AUROC which means the model is able to separate pneumonia from normal.
And also AUPRC, which means a beter precision-recall trade-off.

In [24]:
# DenseNet121 Stage 2 — Screening operating point ONLY
densenet_screen_res = evaluate(
    model,
    test_generator,
    name="DenseNet121 Stage2",
    threshold=0.38
)

# Add metadata for summary tables
densenet_screen_res["Mode"] = "Screening"
densenet_screen_res["Params"] = model.count_params()

densenet_screen_res



=== DenseNet121 Stage2 @ threshold 0.380 ===
[[139  95]
 [  6 384]]
              precision    recall  f1-score   support

      NORMAL       0.96      0.59      0.73       234
   PNEUMONIA       0.80      0.98      0.88       390

    accuracy                           0.84       624
   macro avg       0.88      0.79      0.81       624
weighted avg       0.86      0.84      0.83       624

AUROC: 0.9654 | AUPRC: 0.9772
Macro F1: 0.8086 | Weighted F1: 0.8274
Sensitivity: 0.9846 | Specificity: 0.5940 | Precision: 0.8017


{'Model': 'DenseNet121 Stage2',
 'Flip': None,
 'Class Weights': 'None',
 'Threshold': 0.38,
 'AUROC': 0.96537365768135,
 'AUPRC': 0.9772447496101226,
 'Accuracy': 0.8381410256410257,
 'Macro F1': 0.8086418441116012,
 'Weighted F1': 0.8274249964323775,
 'Sensitivity': 0.9846153846153847,
 'Specificity': 0.594017094017094,
 'Precision': 0.8016701461377871,
 'TN': 139,
 'FP': 95,
 'FN': 6,
 'TP': 384,
 'Mode': 'Screening',
 'Params': 7038529}

In [25]:
import pandas as pd
from pathlib import Path

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

pd.DataFrame([densenet_screen_res]).to_csv(
    out_dir / "densenet_results.csv",
    index=False
)
